# Generate Behavior Dataset With Category Logic

Notebook này sinh dữ liệu hành vi sao cho `product_id`, `action`, `category` và `session intent` có quan hệ logic hơn.
Mục tiêu là để model học được tín hiệu thật hơn thay vì cột `category` bị gán ngẫu nhiên.


In [1]:
import random
from datetime import datetime, timedelta

import numpy as np
import pandas as pd

random.seed(42)
np.random.seed(42)


In [2]:
CATEGORIES = ["Audio", "Laptop", "Monitor", "Accessories", "Storage"]
PRODUCTS_PER_CATEGORY = 20
NUM_PRODUCTS = len(CATEGORIES) * PRODUCTS_PER_CATEGORY

PRODUCT_CATEGORY_MAP = {}
for idx, category in enumerate(CATEGORIES):
    start_id = idx * PRODUCTS_PER_CATEGORY + 1
    end_id = start_id + PRODUCTS_PER_CATEGORY
    for product_id in range(start_id, end_id):
        PRODUCT_CATEGORY_MAP[product_id] = category

product_df = pd.DataFrame({
    "product_id": list(PRODUCT_CATEGORY_MAP.keys()),
    "category": list(PRODUCT_CATEGORY_MAP.values()),
})

product_df.head()


,product_id,category
0,1,Audio
1,2,Audio
2,3,Audio
3,4,Audio
4,5,Audio


In [3]:
SESSION_TEMPLATES = {
    "Audio": [
        ["view", "click", "add_to_cart", "purchase"],
        ["search", "view", "click", "wishlist"],
        ["view", "click", "purchase", "review"],
    ],
    "Laptop": [
        ["search", "view", "click", "add_to_cart"],
        ["view", "click", "wishlist", "purchase"],
        ["search", "view", "click", "remove_from_cart"],
    ],
    "Monitor": [
        ["view", "view", "click", "purchase"],
        ["search", "view", "click", "wishlist"],
        ["view", "click", "add_to_cart", "remove_from_cart"],
    ],
    "Accessories": [
        ["view", "click", "add_to_cart", "purchase", "review"],
        ["view", "wishlist", "purchase"],
        ["search", "view", "click", "add_to_cart"],
    ],
    "Storage": [
        ["search", "view", "click", "purchase"],
        ["view", "click", "add_to_cart", "purchase"],
        ["search", "view", "click", "remove_from_cart"],
    ],
}

ACTION_CATEGORY_WEIGHTS = {
    "view": {"primary": 0.72, "secondary": 0.20, "cross": 0.08},
    "search": {"primary": 0.60, "secondary": 0.30, "cross": 0.10},
    "click": {"primary": 0.80, "secondary": 0.15, "cross": 0.05},
    "add_to_cart": {"primary": 0.90, "secondary": 0.08, "cross": 0.02},
    "purchase": {"primary": 0.92, "secondary": 0.06, "cross": 0.02},
    "wishlist": {"primary": 0.85, "secondary": 0.10, "cross": 0.05},
    "remove_from_cart": {"primary": 0.88, "secondary": 0.08, "cross": 0.04},
    "review": {"primary": 0.95, "secondary": 0.04, "cross": 0.01},
}


In [4]:
def category_products(category):
    return product_df.loc[product_df["category"] == category, "product_id"].tolist()


def choose_secondary_category(primary_category):
    candidates = [c for c in CATEGORIES if c != primary_category]
    return random.choice(candidates)


def choose_product(primary_category, secondary_category, action, last_product_id=None):
    weights = ACTION_CATEGORY_WEIGHTS[action]
    bucket = random.choices(
        population=["primary", "secondary", "cross"],
        weights=[weights["primary"], weights["secondary"], weights["cross"]],
        k=1,
    )[0]

    if action in {"add_to_cart", "purchase", "remove_from_cart", "review"} and last_product_id is not None:
        return last_product_id

    if bucket == "primary":
        chosen_category = primary_category
    elif bucket == "secondary":
        chosen_category = secondary_category
    else:
        chosen_category = random.choice(CATEGORIES)

    return random.choice(category_products(chosen_category))


In [5]:
def generate_user_sequence(user_id, num_sessions=None):
    sequence = []
    session_count = num_sessions or random.randint(3, 10)
    cursor_time = datetime.now()

    for session_id in range(session_count):
        primary_category = random.choice(CATEGORIES)
        secondary_category = choose_secondary_category(primary_category)
        pattern = random.choice(SESSION_TEMPLATES[primary_category])
        last_product_id = None

        for step, action in enumerate(pattern):
            product_id = choose_product(
                primary_category=primary_category,
                secondary_category=secondary_category,
                action=action,
                last_product_id=last_product_id,
            )
            last_product_id = product_id
            sequence.append({
                "user_id": user_id,
                "product_id": product_id,
                "action": action,
                "timestamp": cursor_time + timedelta(minutes=session_id, seconds=step),
                "category": PRODUCT_CATEGORY_MAP[product_id],
                "session_primary_category": primary_category,
            })

        cursor_time += timedelta(minutes=5)

    return sequence


In [6]:
data = []
NUM_USERS = 500

for user_id in range(1, NUM_USERS + 1):
    data.extend(generate_user_sequence(user_id))

df = pd.DataFrame(data)
df = df.sort_values(["user_id", "timestamp"]).reset_index(drop=True)
df.head(20)


,user_id,product_id,action,timestamp,category,session_primary_category
0,1,4,view,2026-04-22 10:25:14.750782,Audio,Audio
1,1,18,click,2026-04-22 10:25:15.750782,Audio,Audio
2,1,18,add_to_cart,2026-04-22 10:25:16.750782,Audio,Audio
3,1,18,purchase,2026-04-22 10:25:17.750782,Audio,Audio
4,1,20,view,2026-04-22 10:31:14.750782,Audio,Audio
5,1,7,click,2026-04-22 10:31:15.750782,Audio,Audio
6,1,7,add_to_cart,2026-04-22 10:31:16.750782,Audio,Audio
7,1,7,purchase,2026-04-22 10:31:17.750782,Audio,Audio
8,1,61,view,2026-04-22 10:37:14.750782,Accessories,Accessories
9,1,66,wishlist,2026-04-22 10:37:15.750782,Accessories,Accessories


In [7]:
print(df["action"].value_counts(normalize=True).round(4))
print()
print(df["category"].value_counts(normalize=True).round(4))
print()
print(pd.crosstab(df["action"], df["category"], normalize="index").round(3))


action
view                0.2659
click               0.2329
purchase            0.1342
search              0.1139
add_to_cart         0.1010
wishlist            0.0653
remove_from_cart    0.0512
review              0.0355
Name: proportion, dtype: float64

category
Audio          0.2070
Accessories    0.2035
Laptop         0.2000
Monitor        0.1983
Storage        0.1911
Name: proportion, dtype: float64

category          Accessories  Audio  Laptop  Monitor  Storage
action                                                        
add_to_cart             0.288  0.179   0.173    0.186    0.175
click                   0.151  0.216   0.215    0.211    0.207
purchase                0.250  0.251   0.137    0.133    0.230
remove_from_cart        0.045  0.039   0.312    0.318    0.286
review                  0.416  0.436   0.048    0.044    0.057
search                  0.169  0.165   0.265    0.165    0.236
view                    0.198  0.197   0.193    0.227    0.184
wishlist               

In [8]:
export_df = df[["user_id", "product_id", "action", "timestamp", "category"]].copy()
export_df.to_csv("data_user500.csv", index=False)
export_df.head()


,user_id,product_id,action,timestamp,category
0,1,4,view,2026-04-22 10:25:14.750782,Audio
1,1,18,click,2026-04-22 10:25:15.750782,Audio
2,1,18,add_to_cart,2026-04-22 10:25:16.750782,Audio
3,1,18,purchase,2026-04-22 10:25:17.750782,Audio
4,1,20,view,2026-04-22 10:31:14.750782,Audio
